In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn

### 1. 数据读取

In [11]:
base = pd.read_csv('../Dataset/ml-100k/u1.base',sep='\t',names=['user_id','item_id','rate','timestamp'])
test = pd.read_csv('../Dataset/ml-100k/u1.test',sep='\t',names=['user_id','item_id','rate','timestamp'])
n_users = base['user_id'].max()
n_items = base['item_id'].max()

print(n_users) # 943
print(n_items) # 1682

943
1682


### 2. 构建评分矩阵和掩码

In [12]:
import torch
R = np.zeros((n_users,n_items),dtype=np.float32)
M = np.zeros((n_users,n_items),dtype=np.float32)

for row in base.itertuples():
    u = row.user_id - 1
    i = row.item_id - 1
    R[u,i] = row.rate
    M[u,i] = 1.0

R_tensor = torch.from_numpy(R)
M_tensor = torch.from_numpy(M)

### 3. 定义模型

In [13]:
class AutoRec(nn.Module):
    def __init__(self,num,hidden_dim):
        super(AutoRec,self).__init__()
        self.encoder = nn.Linear(num,hidden_dim)
        self.decoder = nn.Linear(hidden_dim,num)
        self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        h = self.sigmoid(self.encoder(x))
        r_hat = self.decoder(h)
        return r_hat

model = AutoRec(num=n_items,hidden_dim=200)


### 4. 损失函数

In [14]:
def masked_mse_loss(pred,target,mask,W,V,lam):
    diff = (pred - target) * mask
    mse = (diff ** 2).sum() / mask.sum()
    reg = lam * (W.norm()**2 + V.norm()**2) / 2
    return mse + reg

### 5. 训练

In [16]:
lamda = 0.001
optimizer = torch.optim.Adam(model.parameters(),lamda)
num_epoches = 1000

for epoch in range(num_epoches):
    model.train()
    optimizer.zero_grad()

    pred = model(R_tensor)
    loss = masked_mse_loss(
        pred,R_tensor,M_tensor,
        model.encoder.weight,model.decoder.weight,
        lamda
    )
    loss.backward()
    optimizer.step()

    if(epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1},Loss: {loss.item():.4f}")

Epoch 20,Loss: 1.7858
Epoch 40,Loss: 1.4460
Epoch 60,Loss: 1.2975
Epoch 80,Loss: 1.2523
Epoch 100,Loss: 1.2229
Epoch 120,Loss: 1.1968
Epoch 140,Loss: 1.1713
Epoch 160,Loss: 1.1463
Epoch 180,Loss: 1.1223
Epoch 200,Loss: 1.0998
Epoch 220,Loss: 1.0789
Epoch 240,Loss: 1.0596
Epoch 260,Loss: 1.0417
Epoch 280,Loss: 1.0254
Epoch 300,Loss: 1.0104
Epoch 320,Loss: 0.9963
Epoch 340,Loss: 0.9833
Epoch 360,Loss: 0.9715
Epoch 380,Loss: 0.9605
Epoch 400,Loss: 0.9504
Epoch 420,Loss: 0.9410
Epoch 440,Loss: 0.9320
Epoch 460,Loss: 0.9230
Epoch 480,Loss: 0.9147
Epoch 500,Loss: 0.9071
Epoch 520,Loss: 0.8999
Epoch 540,Loss: 0.8929
Epoch 560,Loss: 0.8855
Epoch 580,Loss: 0.8784
Epoch 600,Loss: 0.8717
Epoch 620,Loss: 0.8653
Epoch 640,Loss: 0.8590
Epoch 660,Loss: 0.8529
Epoch 680,Loss: 0.8469
Epoch 700,Loss: 0.8410
Epoch 720,Loss: 0.8351
Epoch 740,Loss: 0.8293
Epoch 760,Loss: 0.8235
Epoch 780,Loss: 0.8180
Epoch 800,Loss: 0.8125
Epoch 820,Loss: 0.8071
Epoch 840,Loss: 0.8019
Epoch 860,Loss: 0.7969
Epoch 880,Loss:

### 6. 提取训练后的参数

In [17]:
W = model.encoder.weight
b_z = model.encoder.bias
V = model.decoder.weight
b = model.decoder.bias

print("W shape: ",W.shape)
print("b^z shape: ",b_z.shape)
print("V shape: ",V.shape)
print("b shape: ",b.shape)

W shape:  torch.Size([200, 1682])
b^z shape:  torch.Size([200])
V shape:  torch.Size([1682, 200])
b shape:  torch.Size([1682])


### 7. 训练结果的存储与加载

In [19]:
import os 
print(os.getcwd()) # 查看当前文件的所在目录，后续保存的训练结果文件(.pt)也存储在此目录下

# 训练结果的存储
torch.save(model.state_dict(),'AutoRec_Review.pt') # 可以重命名为其他合适的名称

# 训练结果的加载
model = AutoRec(num = n_items,hidden_dim=200)
model.load_state_dict(torch.load('AutoRec_Review.pt'))


d:\Code\Recommender Systems\Review


<All keys matched successfully>

### 8. 测试结果评估

In [21]:
model.eval() # 切换为评估模式
with torch.no_grad():
    R_pred = model(R_tensor).detach().numpy()

errors = []
for row in test.itertuples():
    u = row.user_id - 1
    i = row.item_id - 1
    errors.append((R_pred[u,i] - row.rate) ** 2)

rmse = np.sqrt(np.mean(errors))
print(f"Test RMSE: {rmse:.4f}")



Test RMSE: 0.9562


### 9. 数据分析与可视化

In [2]:
import matplotlib
matplotlib.use('Agg')         
import matplotlib.pyplot as plt

epochs = [500, 1000, 2000]
data = {
    '0.0001': [1.0856, 1.0572, 1.0073],
    '0.001':  [0.9738, 0.9563, 0.9683],
    '0.01':   [1.0186, 1.0161, 1.0151],
    '0.1':    [1.0377, 1.0375, 1.0361],
}
        
plt.figure(figsize=(8, 5))
for lam, rmses in data.items():
    plt.plot(epochs, rmses, marker='o', label=f'lam={lam}')
plt.xlabel('Epoch')
plt.ylabel('Test RMSE')
plt.title('Test RMSE vs Epoch for Different lam')
plt.legend()
plt.grid(True)
plt.savefig('autorec_review_rmse.png', dpi=150, bbox_inches='tight')
plt.close()
print("图片已保存")

图片已保存
